<a href="https://colab.research.google.com/github/alouwyck/Workshop_UM6P_2026/blob/main/Day2/topic_2_3_4_local_search_sa_ga.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Metaheuristics for the Travelling Salesman Problem (TSP)
## From Local Search to Memetic Genetic Algorithms

**What you will learn in this notebook**

This notebook walks you through a progressive hierarchy of optimisation methods applied to the symmetric Travelling Salesman Problem (TSP):

1. **Local Search (2-opt)** — the foundation: given a tour, keep improving it by swapping edges until no single swap helps.
2. **Simulated Annealing** — escape local optima by occasionally accepting worse solutions, with a decreasing probability controlled by a "temperature" schedule.
3. **Genetic Algorithm** — maintain a *population* of tours and combine them via crossover and mutation, mimicking biological evolution.
4. **Memetic GA** — the best of both worlds: a GA where every offspring is polished by local search before re-entering the population.

Each section contains **code**, **step-by-step explanations**, **visualisations**, and **exercises**. By the end you should be able to implement, tune, and compare all four families of metaheuristics on TSP instances of moderate size.

**Prerequisites:** basic Python, NumPy, understanding of graphs and permutations.


In [ ]:
# Imports & utilities
from __future__ import annotations
import math, random, time
from dataclasses import dataclass
from typing import List, Tuple, Callable, Optional, Dict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

random.seed(123)
np.random.seed(123)

def set_seeds(seed: int = 123):
    random.seed(seed)
    np.random.seed(seed)

---
## 1 · Building Blocks: Instance Generation, Evaluation, and the 2-opt Move

Before diving into algorithms, we need three building blocks:
1. **Instance generation** — create random Euclidean TSP instances (cities as $(x,y)$ points).
2. **Tour evaluation** — compute the total length of a tour by summing edge weights around the cycle.
3. **The 2-opt neighbourhood** — the set of tours reachable by reversing one contiguous segment, and the $O(1)$ delta trick that makes it fast.


### Instance generation & tour evaluation

A **Euclidean TSP instance** is a set of $n$ cities, each with coordinates $(x_i, y_i)$, and the distance between two cities $i$ and $j$ is the Euclidean distance $d_{ij} = \sqrt{(x_i - x_j)^2 + (y_i - y_j)^2}$. We precompute the full $n \times n$ symmetric distance matrix $D$ so that any edge lookup is $O(1)$.

A **tour** (feasible solution) is a permutation $\pi = (\pi_0, \pi_1, \dots, \pi_{n-1})$ of city indices. The tour visits city $\pi_0$, then $\pi_1$, ..., then $\pi_{n-1}$, and returns to $\pi_0$. Its **length** (cost) is:

$$\text{cost}(\pi) = \sum_{i=0}^{n-1} d_{\pi_i,\, \pi_{(i+1) \bmod n}}$$

The three functions below handle instance creation (`generate_euclidean_points`), distance matrix computation (`euclidean_dist_matrix`), and tour evaluation (`tour_length`).


In [ ]:
def generate_euclidean_points(n: int, xrange=(0.0, 100.0), yrange=(0.0, 100.0), seed: Optional[int]=None):
    if seed is not None:
        set_seeds(seed)
    xs = np.random.uniform(xrange[0], xrange[1], size=n)
    ys = np.random.uniform(yrange[0], yrange[1], size=n)
    return np.vstack([xs, ys]).T  # shape (n,2)

In [ ]:
def euclidean_dist_matrix(pts: np.ndarray) -> np.ndarray:
    n = len(pts)
    D = np.zeros((n, n), dtype=float)
    for i in range(n):
        for j in range(i+1, n):
            d = math.hypot(pts[i,0]-pts[j,0], pts[i,1]-pts[j,1])
            D[i,j] = D[j,i] = d
    return D

In [ ]:
def tour_length(cost: np.ndarray, tour: List[int]) -> float:
    s = 0.0
    n = len(tour)
    for i in range(n):
        a = tour[i]
        b = tour[(i+1) % n]
        s += cost[a, b]
    return s

### 2-opt move & delta evaluation in $O(1)$

The **2-opt neighbourhood** is the most widely used neighbourhood for TSP. A 2-opt move takes a tour $\pi$ and two indices $i < j$, then **reverses** the segment $[\pi_i, \pi_{i+1}, \dots, \pi_j]$. Geometrically, this removes two edges from the tour and reconnects the resulting two paths in the only other possible way:

```
Before:  ... → π_{i-1} → π_i → π_{i+1} → ... → π_j → π_{j+1} → ...
                  ↓ remove edges (π_{i-1}, π_i) and (π_j, π_{j+1})
After:   ... → π_{i-1} → π_j → π_{j-1} → ... → π_i → π_{j+1} → ...
                  ↑ add edges (π_{i-1}, π_j) and (π_i, π_{j+1})
```

**The $O(1)$ delta trick.** Instead of recomputing the entire tour length after each move (which costs $O(n)$), we only need to look at the two edges removed and the two edges added:

$$\Delta = \bigl[d(\pi_{i-1}, \pi_j) + d(\pi_i, \pi_{j+1})\bigr] - \bigl[d(\pi_{i-1}, \pi_i) + d(\pi_j, \pi_{j+1})\bigr]$$

If $\Delta < 0$ the move improves the tour. This constant-time evaluation is critical because the number of 2-opt neighbours is $O(n^2)$ — without the delta trick, each full neighbourhood scan would cost $O(n^3)$ instead of $O(n^2)$.

**Index conventions.** We use $1 \le i < j \le n{-}1$ (never break the "start edge" $\pi_0 \to \pi_1$) to avoid redundant moves in the cyclic tour representation.


In [ ]:
def two_opt_delta(cost: np.ndarray, tour: List[int], i: int, j: int) -> float:
    """Compute delta = new_length - old_length for reversing segment [i:j] (inclusive).
    Assumes indices i < j and 1 <= i < j < n (no break at start to keep a canonical form).
    We avoid breaking the start edge to handle the tour cyclically.
    """
    n = len(tour)
    a, b = tour[i-1], tour[i]
    c, d = tour[j], tour[(j+1) % n]
    old = cost[a, b] + cost[c, d]
    new = cost[a, c] + cost[b, d]
    return new - old

In [ ]:
def apply_two_opt(tour: List[int], i: int, j: int) -> None:
    """In-place reverse of segment [i:j] inclusive."""
    tour[i:j+1] = reversed(tour[i:j+1])

In [ ]:
def all_two_opt_pairs(n: int):
    """Yield (i, j) for 2-opt with the standard constraints:
    1 <= i < j < n and j - i >= 1 (avoid adjacent edges that produce no change).
    We also avoid reversing around the depot break (index 0).
    """
    for i in range(1, n-1):
        for j in range(i+1, n-0):
            # Skip if reversing just swaps adjacent edges with zero net effect
            if j == i:
                continue
            yield i, j

### Random Search baseline

Before studying intelligent search methods, it is useful to establish a **baseline**: how well does pure random sampling do?

`random_search_tsp` generates `budget` random permutations and keeps track of the best one found. It has no notion of neighbourhood or improvement — each tour is sampled independently and uniformly at random.

**Why include this?** It gives us a lower bar: any decent local search should beat random search by a wide margin. If it doesn't, something is wrong with our implementation.


In [ ]:
def random_tour(n: int) -> List[int]:
    t = list(range(n))
    np.random.shuffle(t)
    return t

In [ ]:
def random_search_tsp(cost: np.ndarray, budget: int, record_history: bool = True, seed: Optional[int]=None):
    if seed is not None:
        set_seeds(seed)
    n = cost.shape[0]
    best = random_tour(n)
    best_val = tour_length(cost, best)
    history = [(0, best_val)] if record_history else None
    t0 = time.time()
    for it in range(1, budget+1):
        cand = random_tour(n)
        val = tour_length(cost, cand)
        if val < best_val:
            best, best_val = cand, val
        if record_history and (it % max(1, budget//50) == 0):
            history.append((it, best_val))
    return best, best_val, history, time.time() - t0

## 2 · Local Search: First-Improvement versus Best-Improvement (2-opt)

**Local search** starts from an initial solution and repeatedly moves to a better neighbour until no improving neighbour exists — at which point we have reached a **local optimum** with respect to the 2-opt neighbourhood.

The two classic pivot rules differ in how they select the improving move:

| Pivot rule | Strategy | Pros | Cons |
|------------|----------|------|------|
| **First-Improvement** | Accept the *first* improving 2-opt move found (pairs are scanned in random order) | Faster per iteration; randomisation helps explore | May miss the best move available |
| **Best-Improvement** | Scan *all* $O(n^2)$ pairs, pick the one with the largest improvement $|\Delta|$ | Steepest descent — each step is optimal | Slower per iteration; deterministic, so always converges to the same local optimum from a given start |

**Key observations:**
- Both methods **always terminate** because every accepted move strictly decreases the tour length, and there are finitely many tours.
- The local optimum reached **depends on the starting tour**. Neither method guarantees finding the global optimum.
- First-improvement typically needs more iterations but each is cheaper; in practice the two methods often reach similar-quality solutions.
- The `record_history` flag lets us track convergence for plotting.


In [ ]:
def ls_first_improvement_2opt(cost: np.ndarray, tour: List[int], max_iters: int = 10**9, record_history: bool=True):
    n = len(tour)
    curr_val = tour_length(cost, tour)
    history = [(0, curr_val)] if record_history else None
    t0 = time.time()
    it = 0
    while it < max_iters:
        it += 1
        improved = False
        # random order of pairs to reduce bias
        pairs = list(all_two_opt_pairs(n))
        random.shuffle(pairs)
        for (i, j) in pairs:
            d = two_opt_delta(cost, tour, i, j)
            if d < 0.0:  # improving
                apply_two_opt(tour, i, j)
                curr_val += d
                improved = True
                break
        if record_history and (it % 1 == 0):
            history.append((it, curr_val))
        if not improved:
            break
    return tour, curr_val, history, time.time() - t0

**Code walkthrough — `ls_first_improvement_2opt`:** The outer `while` loop represents one full "pass" through the neighbourhood. Inside, we generate all valid $(i,j)$ pairs, shuffle them (so the first improving move found is random each pass), and compute $\Delta$ for each. As soon as we find $\Delta < 0$, we apply the move, update the current cost, and break to start a new pass. If a full pass finds no improvement, we've reached a local optimum and stop.

**Code walkthrough — `ls_best_improvement_2opt`:** Similar structure, but instead of breaking at the first improvement, we scan *all* pairs, track the best $\Delta$ and its $(i,j)$, and only apply the single best move at the end of each pass. This is deterministic — no shuffling needed.


In [ ]:
def ls_best_improvement_2opt(cost: np.ndarray, tour: List[int], max_iters: int = 10**9, record_history: bool=True):
    n = len(tour)
    curr_val = tour_length(cost, tour)
    history = [(0, curr_val)] if record_history else None
    t0 = time.time()
    it = 0
    while it < max_iters:
        it += 1
        best_delta = 0.0
        best_pair = None
        for (i, j) in all_two_opt_pairs(n):
            d = two_opt_delta(cost, tour, i, j)
            if d < best_delta:
                best_delta = d
                best_pair = (i, j)
        if best_pair is None:
            break  # local optimum
        apply_two_opt(tour, best_pair[0], best_pair[1])
        curr_val += best_delta
        if record_history:
            history.append((it, curr_val))
    return tour, curr_val, history, time.time() - t0

### Multi-start wrapper & experiment helpers

Since local search always converges to a local optimum (which depends on the starting tour), a simple but effective strategy is **multi-start**: run the local search $R$ times from different random starting tours and keep the overall best solution. This does not change the algorithm itself — it just gives you $R$ independent shots at finding a good local optimum.

We also implement a **nearest-neighbour heuristic** as an alternative starting tour. Starting from a given city, it greedily visits the nearest unvisited city at each step. This produces a decent initial tour (typically 15–25% above optimal) much faster than local search, and can be used to "seed" more advanced methods later.

The `run_experiment_tsp` function automates comparisons across multiple problem sizes and trials, collecting results into a DataFrame for analysis.


In [ ]:
def nearest_neighbor_tour(cost: np.ndarray, start: int = 0) -> List[int]:
    n = cost.shape[0]
    unvisited = set(range(n))
    tour = [start]
    unvisited.remove(start)
    while unvisited:
        last = tour[-1]
        nxt = min(unvisited, key=lambda j: cost[last, j])
        tour.append(nxt)
        unvisited.remove(nxt)
    return tour

In [ ]:
def multistart_ls(cost: np.ndarray, runs: int, pivot: str = "first", start_rule: str="random"):
    best_val = float("inf")
    best_tour = None
    total_time = 0.0
    for r in range(runs):
        if start_rule == "random":
            tour0 = random_tour(cost.shape[0])
        else:
            tour0 = nearest_neighbor_tour(cost)
        if pivot == "first":
            tour, val, hist, t = ls_first_improvement_2opt(cost, tour0, record_history=False)
        else:
            tour, val, hist, t = ls_best_improvement_2opt(cost, tour0, record_history=False)
        total_time += t
        if val < best_val:
            best_val, best_tour = val, tour[:]
    return best_tour, best_val, total_time

In [ ]:
def run_experiment_tsp(ns=(50, 80), trials=3, runs=1, seed=123, compare=("random","first","best")) -> pd.DataFrame:
    set_seeds(seed)
    rows = []
    for n in ns:
        for tr in range(trials):
            pts = generate_euclidean_points(n, seed=seed+tr)
            C = euclidean_dist_matrix(pts)
            # Random Search baseline (budget proportional to n^2)
            if "random" in compare:
                budget = max(200, n*n//2)
                _, val, _, t = random_search_tsp(C, budget=budget, seed=seed+tr, record_history=False)
                rows.append(dict(n=n, trial=tr, method="random_search", time_sec=t, best=val))
            if "first" in compare:
                _, val, t = multistart_ls(C, runs=runs, pivot="first")[:3]
                rows.append(dict(n=n, trial=tr, method="ls_first_2opt_ms", time_sec=t, best=val))
            if "best" in compare:
                _, val, t = multistart_ls(C, runs=runs, pivot="best")[:3]
                rows.append(dict(n=n, trial=tr, method="ls_best_2opt_ms", time_sec=t, best=val))
    return pd.DataFrame(rows)

In [ ]:
def plot_history(history, title="Convergence (best-so-far)"):
    its = [h[0] for h in history]
    vals = [h[1] for h in history]
    plt.figure()
    plt.plot(its, vals)
    plt.xlabel("Iteration")
    plt.ylabel("Tour length (best-so-far)")
    plt.title(title)
    plt.show()

### Quick test

Let's verify everything works on a small 40-city instance. We generate a random Euclidean instance, create a random starting tour, and run both first-improvement and best-improvement LS from the *same* starting tour. This lets us directly compare their convergence behaviour.

**What to look for in the plots:**
- Both methods should reduce the tour length dramatically in the first few iterations.
- First-improvement should take more iterations (each cheap), best-improvement fewer (each expensive).
- They may or may not converge to the same local optimum — this depends on the specific starting tour and instance.


In [ ]:
# Generate a small instance and compare first vs best improvement
pts = generate_euclidean_points(40, seed=42)
C = euclidean_dist_matrix(pts)

t0 = random_tour(len(C))
_, v1, h1, t1 = ls_first_improvement_2opt(C, t0[:], record_history=True)
_, v2, h2, t2 = ls_best_improvement_2opt(C, t0[:], record_history=True)

print("First-impr value/time:", round(v1,2), "/", round(t1,3), "sec")
print("Best-impr value/time:", round(v2,2), "/", round(t2,3), "sec")

plot_history(h1, title="First-Improvement 2-opt")
plot_history(h2, title="Best-Improvement 2-opt")


### Exercises — Local Search

**Exercise 1.1 — Neighbourhood size.**
How many distinct 2-opt moves exist for a tour of length $n$? Derive the formula from the index constraints $1 \le i < j \le n{-}1$. Verify your formula by counting the output of `all_two_opt_pairs(n)` for $n \in \{10, 20, 50\}$.

**Exercise 1.2 — Sensitivity to starting tour.**
Run `ls_first_improvement_2opt` from 20 different random starting tours on the same 40-city instance (use `random_tour(40)` with different seeds). Plot a histogram of the 20 final tour lengths. What is the gap between the best and worst local optima? What does this tell you about the *fitness landscape*?

**Exercise 1.3 — First vs Best: wall-clock time.**
For $n \in \{20, 40, 60, 80, 100\}$, run both pivot rules from the same random start and record the wall-clock time and final tour length. Plot time vs $n$ for both methods. Which one is faster in practice? Does the quality gap grow with $n$?

**Exercise 1.4 — Multi-start effectiveness.**
Using $n = 60$, run `multistart_ls` with $R \in \{1, 3, 5, 10, 20, 50\}$ restarts (first-improvement). Plot the best tour length found vs $R$. At what point do additional restarts give diminishing returns?

**Exercise 1.5 — Or-opt neighbourhood (advanced).**
The **Or-opt** neighbourhood is an alternative to 2-opt: instead of reversing a segment, it *relocates* a segment of 1, 2, or 3 consecutive cities to a different position in the tour. Implement an `or_opt_delta` function (hint: you need to remove the segment from its current position and re-insert it elsewhere — this involves 3 edges removed and 3 edges added, so $\Delta$ can still be computed in $O(1)$). Compare Or-opt local search to 2-opt on the 40-city instance.


---
## 3 · Simulated Annealing (SA) with 2-opt

**The fundamental problem with local search** is that it gets trapped in local optima. Simulated Annealing addresses this by allowing the search to occasionally accept **worse** solutions, giving it a chance to escape and explore new regions of the solution space.

The acceptance rule is the **Metropolis criterion**: given a candidate move with cost change $\Delta = \text{new cost} - \text{current cost}$:
- If $\Delta \le 0$ (improvement): **always accept**.
- If $\Delta > 0$ (worsening): accept with probability $\Pr(\text{accept}) = e^{-\Delta/T}$, where $T > 0$ is the current *temperature*.

**Intuition:** At high temperature, $e^{-\Delta/T} \approx 1$ for moderate $\Delta$, so most uphill moves are accepted — the search is *exploratory*. As $T$ decreases, only small worsenings are tolerated, and eventually the search behaves like greedy local search (*intensification*).

**Geometric cooling schedule:** The simplest and most common cooling schedule sets $T_k = \alpha^k \cdot T_0$ where:
- $T_0$ is the **initial temperature** (calibrated to accept a target fraction $\rho_0 \approx 0.80$ of uphill moves at the start),
- $\alpha \in (0, 1)$ is the **cooling rate** (typically 0.90–0.99),
- The total budget is $B = K \cdot L$ Metropolis trials, split into $K$ temperature stages of $L$ trials each.

**$T_0$ calibration:** Rather than guessing $T_0$, we *sample* random uphill deltas from the initial solution and set $T_0 = -\bar{\Delta} / \ln(\rho_0)$. This ensures the initial acceptance rate is approximately $\rho_0$.


In [ ]:
# ── T0 calibration ─────────────────────────────────────────────────────────────
def calibrate_T0(cost: np.ndarray, tour: List[int],
                 M: int = 300,
                 target_acceptance: float = 0.80,
                 rng: Optional[random.Random] = None) -> float:
    """
    Sample M random 2-opt moves, collect the uphill deltas (Δ > 0),
    and return T0 via the proxy formula:

        T0 = -mean_delta / ln(target_acceptance)

    This ensures mean(exp(-Δ/T0)) ≈ target_acceptance over the uphill sample.
    """
    if rng is None:
        rng = random.Random()
    n = len(tour)
    pairs = list(all_two_opt_pairs(n))
    uphill = []
    for _ in range(M):
        i, j = rng.choice(pairs)
        d = two_opt_delta(cost, tour, i, j)
        if d > 0.0:
            uphill.append(d)
    if not uphill:           # degenerate: no uphill moves found → fallback
        return 1.0
    mean_delta = sum(uphill) / len(uphill)
    return -mean_delta / math.log(target_acceptance)

**Code walkthrough — `calibrate_T0`:** We sample $M$ random 2-opt moves from the initial tour and collect only the uphill deltas ($\Delta > 0$). Their mean $\bar{\Delta}$ characterises the typical "hill height" in the landscape. We then solve $e^{-\bar{\Delta}/T_0} = \rho_0$ for $T_0$, yielding $T_0 = -\bar{\Delta}/\ln(\rho_0)$. For the default $\rho_0 = 0.80$, this gives $T_0 \approx 4.48 \cdot \bar{\Delta}$.

**Code walkthrough — `sa_tsp`:** The outer loop iterates over $K$ temperature stages. At each stage, the inner loop performs $L$ Metropolis trials: pick a random 2-opt move, compute $\Delta$; if $\Delta < 0$ accept immediately, otherwise accept with probability $e^{-\Delta/T}$. After each stage, cool: $T \leftarrow \alpha \cdot T$. The algorithm tracks the overall best solution seen (which may differ from the current solution, since SA can accept worsenings).


In [ ]:
# ── Core SA algorithm ──────────────────────────────────────────────────────────
def sa_tsp(cost: np.ndarray,
           tour: List[int],
           T0: Optional[float] = None,
           alpha: float = 0.95,
           L: Optional[int] = None,
           max_stages: int = 200,
           target_acceptance: float = 0.80,
           seed: Optional[int] = None,
           record_history: bool = True):
    """
    Simulated Annealing for TSP using random 2-opt moves + geometric cooling.

    Parameters
    ----------
    cost              : n×n symmetric distance matrix
    tour              : initial tour (not modified; a copy is used internally)
    T0                : initial temperature; auto-calibrated if None
    alpha             : geometric cooling factor  (0 < alpha < 1)
    L                 : iterations per temperature stage; default = 10·n
    max_stages        : number of cooling stages  K  (budget B = K·L)
    target_acceptance : uphill acceptance rate ρ₀ used for T0 calibration
    seed              : RNG seed for full reproducibility
    record_history    : if True, append (total_iters, best_val) after each stage

    Returns
    -------
    best_tour, best_val, history, elapsed_sec
    """
    rng = random.Random(seed)
    n   = len(tour)
    if L is None:
        L = 10 * n

    # ── calibrate T0 ────────────────────────────────────────────────────────
    if T0 is None:
        T0 = calibrate_T0(cost, tour, M=300,
                          target_acceptance=target_acceptance, rng=rng)

    # ── initialise ──────────────────────────────────────────────────────────
    pairs     = list(all_two_opt_pairs(n))   # precompute once
    tour      = tour[:]                       # work on a local copy
    curr_val  = tour_length(cost, tour)
    best_tour = tour[:]
    best_val  = curr_val
    T         = T0

    history  = [(0, best_val)] if record_history else None
    t_start  = time.time()

    # ── outer loop: temperature stages ──────────────────────────────────────
    for stage in range(1, max_stages + 1):
        accepted = 0
        # ── inner loop: Metropolis trials at fixed T ─────────────────────
        for _ in range(L):
            i, j  = rng.choice(pairs)
            delta = two_opt_delta(cost, tour, i, j)
            if delta <= 0.0 or rng.random() < math.exp(-delta / T):
                apply_two_opt(tour, i, j)
                curr_val += delta
                accepted += 1
                if curr_val < best_val:
                    best_val  = curr_val
                    best_tour = tour[:]
        T *= alpha                            # geometric cooling
        if record_history:
            history.append((stage * L, best_val))

    return best_tour, best_val, history, time.time() - t_start

### T$_0$ calibration demo

The calibration procedure is best understood by working through a concrete example. We take the same 40-city instance from the smoke test, sample 500 random 2-opt moves, keep only the uphill ones ($\Delta > 0$), and compute their mean. The formula $T_0 = -\bar{\Delta}/\ln(0.80)$ then gives us the initial temperature.

**Verification:** we check that $\frac{1}{|\mathcal{U}|}\sum_{\Delta \in \mathcal{U}} e^{-\Delta/T_0} \approx 0.80$, confirming the calibration works. The histogram below shows the distribution of uphill deltas — notice it is right-skewed, meaning most uphill moves are small (the landscape is relatively smooth) but a few are large.


In [ ]:
# Use the 40-city instance from the smoke test above (pts, C, t0)
rng_demo = random.Random(42)
n_demo   = len(C)
pairs_demo = list(all_two_opt_pairs(n_demo))

In [ ]:
# ── Sample 500 random 2-opt moves and keep uphill deltas ──────────────────────
uphill_deltas = [d for _ in range(500)
                 for d in [two_opt_delta(C, t0, *rng_demo.choice(pairs_demo))]
                 if d > 0.0]

T0_cal = calibrate_T0(C, t0, M=500, target_acceptance=0.80, rng=random.Random(42))
mean_acc_empirical = sum(math.exp(-d / T0_cal) for d in uphill_deltas) / len(uphill_deltas)

print(f"Uphill deltas sampled  : {len(uphill_deltas)}")
print(f"Mean uphill Δ          : {sum(uphill_deltas)/len(uphill_deltas):.3f}")
print(f"Calibrated T₀          : {T0_cal:.3f}")
print(f"Empirical mean acc @ T₀: {mean_acc_empirical:.3f}  (target ρ₀ = 0.80)")

In [ ]:
# Distribution of uphill deltas
plt.figure(figsize=(6, 3))
plt.hist(uphill_deltas, bins=30, color="steelblue", edgecolor="white", density=True)
plt.axvline(sum(uphill_deltas)/len(uphill_deltas), color="orange", lw=1.5,
            label=f"mean Δ = {sum(uphill_deltas)/len(uphill_deltas):.1f}")
plt.xlabel("Uphill Δ  (2-opt, 40 cities)")
plt.ylabel("Density")
plt.title(f"Uphill delta distribution  →  T₀ = {T0_cal:.1f}")
plt.legend()
plt.tight_layout()
plt.show()

### Cooling schedule at a glance

The table below shows how the temperature drops across stages and what that means for Metropolis acceptance. We look at two representative uphill barriers — a *small* one ($\Delta = \bar{\Delta}/2$) and a *large* one ($\Delta = 2\bar{\Delta}$).

**Reading the table:** In the early stages (high $T$), even large worsenings have acceptance probabilities above 50%. By the final stages, only tiny worsenings are accepted — the search has essentially become greedy. This gradual transition from exploration to exploitation is the core mechanism of SA.

**Design choice: $L$ (iterations per stage).** We set $L = 10n$ by default. This gives each temperature level enough Metropolis trials to adequately sample the neighbourhood before cooling. Too small an $L$ means the temperature drops before the search has had time to explore at that level; too large wastes budget at temperatures that may no longer be useful.


In [ ]:
def print_sa_schedule(T0: float, alpha: float, L: int, K: int,
                      delta_small: float, delta_large: float) -> None:
    """
    Print a compact schedule table: stage, temperature, and Metropolis
    acceptance probabilities for two representative uphill deltas.
    """
    header = (f"{'Stage':>6}  {'T':>9}  "
              f"{'p(Δ={:.1f})'.format(delta_small):>12}  "
              f"{'p(Δ={:.1f})'.format(delta_large):>12}")
    print(header)
    print("-" * len(header))
    checkpoints = [0, K//5, 2*K//5, 3*K//5, 4*K//5, K - 1]
    T = T0
    for k in range(K):
        if k in checkpoints:
            p_s = math.exp(-delta_small / T) if T > 1e-12 else 0.0
            p_l = math.exp(-delta_large / T) if T > 1e-12 else 0.0
            print(f"{k:>6}  {T:>9.3f}  {p_s:>12.4f}  {p_l:>12.4f}")
        T *= alpha
    T_final = T0 * alpha**K
    print(f"\n  T_final ≈ {T_final:.4f}   (B = {K*L:,} total iterations)")


# Example parameters matching the 40-city instance
n   = len(C)
L   = 10 * n         # 400
K   = 200            # stages
T0  = T0_cal         # from calibration above
alpha = 0.95
mean_d = sum(uphill_deltas) / len(uphill_deltas)

print(f"Schedule: n={n}, L={L}, K={K}, alpha={alpha}, T0={T0:.2f}\n")
print_sa_schedule(T0, alpha, L, K,
                  delta_small=mean_d / 2,
                  delta_large=mean_d * 2)


### SA convergence — same 40-city instance

We run SA from the *same* initial tour `t0` used in the LS smoke test, so the convergence curves are directly comparable.

In [ ]:
# Run SA (same initial tour as smoke test, alpha=0.95, 300 stages)
sa_tour, sa_val, sa_hist, sa_time = sa_tsp(
    C, t0[:], alpha=0.95, max_stages=300, seed=42)

print("─" * 42)
print(f"{'Method':<22}  {'Cost':>8}  {'Time':>6}")
print("─" * 42)
print(f"{'First-impr. LS':<22}  {v1:>8.2f}  {t1:>5.3f}s")
print(f"{'Best-impr. LS':<22}  {v2:>8.2f}  {t2:>5.3f}s")
print(f"{'SA  (α=0.95, K=300)':<22}  {sa_val:>8.2f}  {sa_time:>5.3f}s")
print("─" * 42)

# Convergence comparison (all three on one figure)
# The LS histories track *passes*, SA histories track *stage × L* — both are
# "number of 2-opt evaluations" so they share the same x-axis interpretation.
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot([h[0] for h in h1], [h[1] for h in h1],
        label="First-impr. LS", color="tab:blue")
ax.plot([h[0] for h in h2], [h[1] for h in h2],
        label="Best-impr. LS",  color="tab:green")
ax.plot([h[0] for h in sa_hist], [h[1] for h in sa_hist],
        label="SA  (α=0.95)",   color="tab:red", lw=1.5)
ax.set_xlabel("Cumulative 2-opt evaluations  (approx.)")
ax.set_ylabel("Best-so-far tour length")
ax.set_title("First-impr. LS vs Best-impr. LS vs SA  (40 cities)")
ax.legend()
plt.tight_layout()
plt.show()


### Effect of cooling rate $\alpha$

The cooling rate $\alpha$ is arguably the most important hyperparameter in SA. It controls how quickly the temperature drops and therefore the balance between exploration and exploitation:

- **Small $\alpha$ (e.g., 0.90):** Temperature drops rapidly — fewer useful stages before the search freezes. Risk: premature convergence to a mediocre local optimum.
- **Large $\alpha$ (e.g., 0.99):** Temperature drops slowly — the search spends many stages at high temperature, accepting almost anything. For a fixed budget $K$, this means less time is spent at the productive intermediate temperatures. Risk: insufficient intensification.
- **Sweet spot (typically 0.93–0.95):** Provides enough exploration early on while still reaching low temperatures within the budget.

We sweep $\alpha \in \{0.90, 0.93, 0.95, 0.97, 0.99\}$ with a fixed stage budget of $K = 300$ to visualise this trade-off. The left panel shows convergence curves, the right panel shows final tour lengths.


In [ ]:
alphas = [0.90, 0.93, 0.95, 0.97, 0.99]
colors = plt.cm.plasma(np.linspace(0.15, 0.85, len(alphas)))

fig, (ax_conv, ax_bar) = plt.subplots(1, 2, figsize=(11, 4))

final_vals = {}
for al, col in zip(alphas, colors):
    _, val, hist, _ = sa_tsp(C, t0[:], alpha=al, max_stages=300, seed=42)
    final_vals[al] = val
    ax_conv.plot([h[0] for h in hist], [h[1] for h in hist],
                 label=f"α={al}", color=col, lw=1.4)
    print(f"α={al}:  final cost = {val:.2f}")

ax_conv.set_xlabel("Cumulative iterations  (stage × L)")
ax_conv.set_ylabel("Best-so-far tour length")
ax_conv.set_title("Convergence for different α  (40 cities)")
ax_conv.legend(title="α", fontsize=8)

# Bar chart of final costs
ax_bar.bar([str(a) for a in alphas], [final_vals[a] for a in alphas],
           color=colors, edgecolor="grey", linewidth=0.5)
ax_bar.set_xlabel("α")
ax_bar.set_ylabel("Final tour length")
ax_bar.set_title("Final cost vs cooling rate α")
ax_bar.set_ylim(min(final_vals.values()) * 0.95, max(final_vals.values()) * 1.02)

plt.tight_layout()
plt.show()


### Multi-start SA and full comparison

Even with the ability to accept uphill moves, a single SA run may still converge to a suboptimal basin of attraction. **Multi-start SA** is the simplest remedy: run SA $R$ times from independent random initial tours, each with its own RNG stream, and keep the best result.

**Why does this help?** Different starting tours place the search in different regions of the solution space. Each SA run then explores its local region thoroughly. By taking the best of $R$ independent explorations, we effectively sample $R$ different attraction basins.

**Trade-off:** Multi-start multiplies the computation time by $R$. An alternative (not implemented here) is to use a *reheating* schedule within a single long run, which can achieve similar diversity without the overhead of re-initialisation.

The experiment below compares four strategies across $n \in \{40, 80\}$ with 5 independent trials each: LS-first, LS-best, single-start SA, and multi-start SA ($\times 3$).


In [ ]:
def multistart_sa(cost: np.ndarray, runs: int = 5,
                  alpha: float = 0.95,
                  L: Optional[int] = None,
                  max_stages: int = 200,
                  seed: Optional[int] = None):
    """
    Run SA from `runs` independent random starts; return the overall best tour.
    Each run gets its own random initial tour and its own RNG stream.
    """
    n   = cost.shape[0]
    rng = random.Random(seed)
    best_val  = float("inf")
    best_tour = None
    total_time = 0.0
    for _ in range(runs):
        init_tour = list(range(n))
        rng.shuffle(init_tour)
        run_seed = rng.randint(0, 2**31)
        tour, val, _, t = sa_tsp(cost, init_tour, alpha=alpha, L=L,
                                  max_stages=max_stages, seed=run_seed,
                                  record_history=False)
        total_time += t
        if val < best_val:
            best_val, best_tour = val, tour
    return best_tour, best_val, total_time

In [ ]:
# ── Full comparison experiment ─────────────────────────────────────────────────
def run_comparison(ns=(40, 80), trials=5, seed=123) -> pd.DataFrame:
    set_seeds(seed)
    rows = []
    for n in ns:
        for tr in range(trials):
            pts_e = generate_euclidean_points(n, seed=seed + tr * 7)
            C_e   = euclidean_dist_matrix(pts_e)
            init  = random_tour(n)   # shared starting tour for LS runs

            # First-improvement LS
            _, val, _, t = ls_first_improvement_2opt(C_e, init[:], record_history=False)
            rows.append(dict(n=n, trial=tr, method="LS-first",    cost=val, time=t))

            # Best-improvement LS
            _, val, _, t = ls_best_improvement_2opt(C_e, init[:], record_history=False)
            rows.append(dict(n=n, trial=tr, method="LS-best",     cost=val, time=t))

            # SA single start (same init tour)
            _, val, _, t = sa_tsp(C_e, init[:], alpha=0.95, max_stages=200, seed=seed+tr)
            rows.append(dict(n=n, trial=tr, method="SA (×1)",     cost=val, time=t))

            # SA multi-start (3 independent runs)
            _, val, t = multistart_sa(C_e, runs=3, alpha=0.95, max_stages=200, seed=seed+tr)
            rows.append(dict(n=n, trial=tr, method="SA-ms (×3)",  cost=val, time=t))

    return pd.DataFrame(rows)

In [ ]:
df = run_comparison(ns=(40, 80), trials=5)

# Summary table: mean and min across trials
summary = (df.groupby(["n", "method"])["cost"]
             .agg(mean="mean", best="min", worst="max")
             .round(1))
print(summary.to_string())

In [ ]:
# Box-plot: solution quality by method and problem size
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=False)
method_order = ["LS-first", "LS-best", "SA (×1)", "SA-ms (×3)"]
colors_box = ["tab:blue", "tab:green", "tab:orange", "tab:red"]

for ax, n_val in zip(axes, [40, 80]):
    sub  = df[df["n"] == n_val]
    data = [sub[sub["method"] == m]["cost"].values for m in method_order]
    bp   = ax.boxplot(data, tick_labels=method_order, patch_artist=True, sym="+")
    for patch, col in zip(bp["boxes"], colors_box):
        patch.set_facecolor(col)
        patch.set_alpha(0.6)
    ax.set_title(f"n = {n_val} cities  (5 trials)")
    ax.set_ylabel("Tour length" if n_val == 40 else "")
    ax.set_xlabel("Method")
    plt.setp(ax.get_xticklabels(), rotation=20, ha="right", fontsize=8)

plt.suptitle("Solution quality: LS vs SA  (lower is better)", y=1.01)
plt.tight_layout()
plt.show()


### Exercises — Simulated Annealing

**Exercise 2.1 — Verify Metropolis acceptance by hand.**
Consider a current tour with cost 500, a candidate move with $\Delta = +12$, and temperature $T = 20$. Compute the acceptance probability. Would the move be accepted if the random number drawn is 0.43? What if $T = 5$?

**Exercise 2.2 — $T_0$ sensitivity.**
Modify `sa_tsp` to accept a manual $T_0$ override. Run SA on the 40-city instance with $T_0 \in \{1, 5, 20, 50, 200\}$ (keeping $\alpha = 0.95$, $K = 200$). Plot the convergence curves. What happens when $T_0$ is too low? Too high? How does the automatically calibrated $T_0$ compare?

**Exercise 2.3 — Iterations per stage $L$.**
Fix $\alpha = 0.95$ and total budget $B = 80{,}000$ iterations. Vary $L \in \{50, 200, 400, 800\}$ (which implies $K = B/L$ stages). Run SA on the 40-city instance for each setting and compare final tour lengths. Is there an optimal $L$? Why might very small or very large $L$ be suboptimal?

**Exercise 2.4 — Reheating (advanced).**
Implement a variant of SA that, after the temperature reaches a floor $T_{\min}$, *reheats* back to $T_0/2$ and cools again. This is called **non-monotone cooling**. Compare single-reheat SA to multi-start SA ($R = 2$) with the same total budget. Which strategy finds better solutions?

**Exercise 2.5 — SA vs LS on larger instances.**
Generate instances with $n \in \{100, 150, 200\}$. For each, run multi-start LS (10 restarts, first-improvement) and single SA ($\alpha = 0.95$, $K = 500$). Report the best tour length and total time. At what problem size does SA start to clearly outperform multi-start LS? (Hint: SA's advantage grows as the landscape becomes more complex.)

**Exercise 2.6 — Adaptive cooling (advanced).**
Instead of a fixed $\alpha$, implement **adaptive cooling** where $\alpha$ is increased (slower cooling) when the acceptance rate drops below a threshold and decreased when it is too high. A simple rule: if the acceptance rate in the current stage is below 0.20, set $\alpha \leftarrow \min(\alpha \cdot 1.01, 0.999)$; if above 0.50, set $\alpha \leftarrow \max(\alpha \cdot 0.99, 0.80)$. Compare this adaptive SA to fixed-$\alpha$ SA.


---
## 4 · Genetic Algorithm (GA) for TSP

**From single-solution to population-based search.** Both LS and SA maintain a *single* current solution. A Genetic Algorithm takes a fundamentally different approach: it maintains a **population** of $N$ candidate tours and evolves the entire population over $G$ generations through operators inspired by biological evolution.

**Why populations help:** By maintaining diverse solutions simultaneously, a GA can explore multiple regions of the solution space in parallel. Crossover (recombination) can combine good "building blocks" from different tours, potentially creating offspring that are better than either parent — something no single-solution method can do.

The three core operators and their TSP-specific implementations:

| Operation | Purpose | TSP specialisation |
|-----------|---------|-------------------|
| **Selection** | Favour fitter individuals for reproduction | Tournament selection (size $k$): pick $k$ random individuals, the best one wins |
| **Crossover** | Recombine two parent tours into offspring | OX or PMX — standard 1-point crossover creates duplicate cities! |
| **Mutation** | Introduce small random perturbations | Inversion (= one 2-opt move) or swap |

Plus **elitism**: the $e$ best individuals from the current generation are copied unchanged to the next, ensuring the best solution never gets lost.

**Canonical GA loop** (generational replacement with elitism):

```
P ← Initialise(N)           # mixed: ~15% nearest-neighbour, rest random
for g = 1 to G:
    P' ← ∅
    while |P'| < N - e:
        p1, p2 ← TournamentSelect(P, k)
        if random() < pc:
            c1, c2 ← Crossover(p1, p2)
        else:
            c1, c2 ← clone(p1), clone(p2)
        Mutate(c1, pm);  Mutate(c2, pm)
        P' ← P' ∪ {c1, c2}
    P ← Top-e(P) ∪ P'       # elitism: keep best e from old P
best ← argmin_{x ∈ P} f(x)
```

**Mixed initialisation (Prins 2004):** Instead of starting with all random tours, we seed ~15% of the population with nearest-neighbour tours (from different starting cities). This gives the GA a head start with reasonably good solutions while maintaining diversity through the random majority.


### Crossover operators

**Why standard crossover fails for TSP.** In binary or real-valued GAs, one-point crossover simply splices two parent strings. But for permutation-based problems like TSP, this produces **invalid tours** with duplicate cities and missing cities. TSP-specific crossover operators must ensure every offspring is a valid permutation.

Two widely used permutation crossover operators:

**OX (Order Crossover, Davis 1985):**
1. Choose a random segment $[a, b]$ in the child; copy it directly from Parent 1.
2. Fill the remaining positions with cities from Parent 2, preserving their *relative order* (scan P2 from position $b{+}1$, wrapping around, skip cities already in the segment).

*What OX preserves:* the absolute positions of cities in the copied segment (from P1) and the relative ordering of the remaining cities (from P2). This is particularly effective for Euclidean TSP where adjacency and ordering matter.

**PMX (Partially Mapped Crossover, Goldberg & Lingle 1985):**
1. Choose a random segment $[a, b]$; copy it from Parent 2 into the child.
2. For each position outside $[a, b]$: try to place P1's city there. If it conflicts (already in the segment), follow the *mapping chain* $P2[k] \to P1[k]$ until a non-conflicting city is found.

*What PMX preserves:* the absolute positions of cities both inside the segment (from P2) and outside (from P1, with conflict resolution). The mapping chain ensures every city appears exactly once.

The step-by-step demos below walk through both operators on the same 8-city example with the same cut points, so you can see exactly how they differ.


In [ ]:
# ── Order Crossover (OX) — Davis (1985) ───────────────────────────────────────
def crossover_ox(p1: List[int], p2: List[int], rng: random.Random) -> List[int]:
    """
    Order Crossover (OX) — Davis (1985).  O(n).

    Algorithm
    ---------
    1. Choose random segment [a, b]; copy p1[a:b+1] directly into child.
    2. Scan p2 from position b+1 (wrapping); insert cities not yet in child
       into the remaining positions of child (also starting from b+1, wrapping).

    Preserves the *relative order* of cities inherited from p2 — the key
    property exploited by Euclidean TSP instances.
    """
    n = len(p1)
    a, b = sorted(rng.sample(range(n), 2))
    child = [None] * n
    child[a:b+1] = p1[a:b+1]          # Step 1: copy segment
    seg = set(child[a:b+1])
    # Step 2: cities from p2 in wrap-order starting at b+1, skipping segment
    fill_cities = [p2[(b + 1 + k) % n] for k in range(n)
                   if p2[(b + 1 + k) % n] not in seg]
    # Positions to fill in same wrap-order, skipping [a:b+1]
    fill_pos    = [(b + 1 + k) % n for k in range(n)
                   if (b + 1 + k) % n < a or (b + 1 + k) % n > b]
    for pos, city in zip(fill_pos, fill_cities):
        child[pos] = city
    return child

In [ ]:
# ── Partially Mapped Crossover (PMX) — Goldberg & Lingle (1985) ───────────────
def crossover_pmx(p1: List[int], p2: List[int], rng: random.Random) -> List[int]:
    """
    Partially Mapped Crossover (PMX) — Goldberg & Lingle (1985).  O(n).

    Algorithm
    ---------
    1. Choose random segment [a, b]; copy p2[a:b+1] into child.
    2. For each position outside [a:b+1]: place p1's city there, but if it
       conflicts (already in segment), follow the segment mapping chain
       p2[k] → p1[k] until a non-conflicting city is found.
    """
    n = len(p1)
    a, b = sorted(rng.sample(range(n), 2))
    child = [None] * n
    child[a:b+1] = p2[a:b+1]          # Step 1: copy p2 segment
    seg_set = set(p2[a:b+1])
    pos_in_p2 = {city: idx for idx, city in enumerate(p2)}
    for k in range(n):                 # Step 2: fill outside positions
        if k < a or k > b:
            city = p1[k]
            while city in seg_set:     # follow mapping chain
                city = p1[pos_in_p2[city]]
            child[k] = city
    return child

**Code walkthrough — `crossover_ox`:** The key insight is the wrap-around scan. After copying P1's segment into positions $[a, b]$, we start scanning P2 from position $b{+}1$ (wrapping to the start when we reach the end). We skip any city already placed in the segment. The remaining cities are placed into the child in the same wrap-around order, starting from position $b{+}1$. This preserves the cyclic relative order from P2.

**Code walkthrough — `crossover_pmx`:** After copying P2's segment, we build a reverse index `pos_in_p2` mapping each city to its position in P2. For each position $k$ outside $[a, b]$, we start with P1's city at that position. If it's already in the segment (conflict), we look up where it sits in P2 (say position $m$), then take P1's city at position $m$ instead. We repeat until we find a city not in the segment — this is the "mapping chain."

Let's trace through both operators step by step on a concrete example:


In [ ]:
# ── Step-by-step OX demo ───────────────────────────────────────────────────────
print("=== Order Crossover (OX) — step-by-step ===\n")
p1_d = [1, 2, 3, 4, 5, 6, 7, 8]
p2_d = [2, 4, 6, 1, 7, 3, 5, 8]
a_d, b_d = 2, 4      # fixed cut points for clarity

seg_d = p1_d[a_d:b_d+1]
seg_set_d = set(seg_d)
print(f"P1            : {p1_d}")
print(f"P2            : {p2_d}")
print(f"Cut [a={a_d}, b={b_d}]  →  segment from P1 = {seg_d}\n")

step1 = ['_'] * 8
step1[a_d:b_d+1] = [str(x) for x in seg_d]
print(f"Step 1 — copy P1[{a_d}:{b_d+1}] into child positions {a_d}–{b_d}:")
print(f"  child = {step1}\n")

scan   = [p2_d[(b_d+1+k) % 8] for k in range(8)]
filled = [c for c in scan if c not in seg_set_d]
fpos   = [(b_d+1+k) % 8 for k in range(8) if (b_d+1+k) % 8 < a_d or (b_d+1+k) % 8 > b_d]
print(f"Step 2 — scan P2 from pos {b_d+1} (wrap): {scan}")
print(f"         keep cities ∉ segment {{{','.join(map(str,seg_d))}}}: {filled}")
print(f"         fill positions {fpos}")

child_d = crossover_ox(p1_d, p2_d, random.Random(0))   # any rng (cut points are random, but result valid)
# Force the fixed cut for display:
child_fixed = [None]*8
child_fixed[a_d:b_d+1] = p1_d[a_d:b_d+1]
for pos, city in zip(fpos, filled):
    child_fixed[pos] = city
print(f"\n  Child (OX)  : {child_fixed}")
print(f"  Valid tour? : {sorted(child_fixed) == list(range(1,9))}")

In [ ]:
print("\n=== PMX demo (same parents, same cut) ===\n")
child_pmx = [None]*8
child_pmx[a_d:b_d+1] = p2_d[a_d:b_d+1]
seg_pmx = set(p2_d[a_d:b_d+1])
pos_p2  = {city: idx for idx, city in enumerate(p2_d)}
for k in range(8):
    if k < a_d or k > b_d:
        city = p1_d[k]
        chain = [city]
        while city in seg_pmx:
            city = p1_d[pos_p2[city]]
            chain.append(city)
        child_pmx[k] = city
        chain_str = " → ".join(map(str, chain))
        print(f"  pos {k}: P1[{k}]={p1_d[k]}  chain: {chain_str}  →  child[{k}]={city}")
print(f"\n  Child (PMX) : {child_pmx}")
print(f"  Valid tour? : {sorted(child_pmx) == list(range(1,9))}")

### Mutation operators

Mutation provides the random variation that prevents the population from converging prematurely. Without mutation, the GA can only recombine existing genetic material — it cannot introduce genuinely new subsequences.

| Operator | Effect | Cost | Connection to LS |
|----------|--------|------|-----------------|
| **Inversion** | Reverse segment $[\pi_i,\ldots,\pi_j]$ | $O(n)$ | Identical to one 2-opt move — bridges GA and LS |
| **Swap** | Exchange two random cities $\pi_i \leftrightarrow \pi_j$ | $O(1)$ | Simpler but less structured; does not correspond to edge-swap |

**Mutation rate $p_m$:** The probability that a given offspring undergoes mutation. A common recommendation is $p_m = 1/n$, which means on average one individual per generation gets mutated. Too high a rate destroys good solutions; too low and the population stagnates.

**Why inversion is preferred for TSP:** Since inversion is equivalent to a 2-opt move, it operates in the same neighbourhood as our local search. This means mutation can make the same kind of improvements that LS would — a useful synergy, especially in the memetic GA we'll see later.


In [ ]:
def mutate_inversion(tour: List[int], pm: float, rng: random.Random) -> None:
    """
    Inversion mutation: reverse a random segment [i, j].
    Identical to one 2-opt move. Modifies tour in-place.
    """
    if rng.random() < pm:
        n = len(tour)
        i, j = sorted(rng.sample(range(n), 2))
        tour[i:j+1] = tour[i:j+1][::-1]

In [ ]:
def mutate_swap(tour: List[int], pm: float, rng: random.Random) -> None:
    """
    Swap mutation: exchange two random cities. O(1). Modifies in-place.
    """
    if rng.random() < pm:
        n = len(tour)
        i, j = rng.sample(range(n), 2)
        tour[i], tour[j] = tour[j], tour[i]

In [ ]:
# ── Mutation demo ──────────────────────────────────────────────────────────────
ex_tour = list(range(1, 9))
rng_m   = random.Random(7)

In [ ]:
inv_tour = ex_tour[:]
i_m, j_m = 2, 5
inv_tour[i_m:j_m+1] = inv_tour[i_m:j_m+1][::-1]
print(f"Inversion mutation  i={i_m}, j={j_m}")
print(f"  Before : {ex_tour}")
print(f"  After  : {inv_tour}   (segment {ex_tour[i_m:j_m+1]} reversed → {inv_tour[i_m:j_m+1]})\n")

In [ ]:
sw_tour = ex_tour[:]
ia, ib  = 1, 6
sw_tour[ia], sw_tour[ib] = sw_tour[ib], sw_tour[ia]
print(f"Swap mutation  positions {ia} ↔ {ib}")
print(f"  Before : {ex_tour}")
print(f"  After  : {sw_tour}")

### Canonical GA implementation

**Tournament selection** works by randomly sampling $k$ individuals from the population and selecting the one with the lowest cost (best fitness). This is simple to implement and provides tuneable selection pressure: larger $k$ increases the probability that the best individual in the tournament is very fit, leading to stronger exploitation. Smaller $k$ gives weaker individuals a better chance, maintaining diversity.

Key parameters and their recommended ranges:

| Parameter | Symbol | Typical range | Effect |
|-----------|--------|--------------|--------|
| Population size | $N$ | 50–200; $\approx n$ is a good start | Diversity vs computational cost per generation |
| Generations | $G$ | 100–500 | Total evolutionary budget |
| Crossover rate | $p_c$ | 0.85–0.95 | Probability of recombination vs cloning |
| Mutation rate | $p_m$ | $1/n$ to 0.05 | Exploration intensity |
| Tournament size | $k$ | 2–5 | Selection pressure (higher = more exploitative) |
| Elites | $e$ | 1–3 | Guaranteed preservation of best solutions |

**Code walkthrough — `ga_tsp`:** The function first creates a mixed initial population (NN greedy + random). Then for each generation: (1) identify and preserve the top $e$ elites, (2) fill the remaining $N - e$ slots by repeatedly selecting two parents via tournament, applying crossover with probability $p_c$, mutating each offspring with probability $p_m$, and adding them to the new population. The best-ever solution is tracked across all generations.


In [ ]:
# ── Tournament selection helper ────────────────────────────────────────────────
def _tournament(population: List[List[int]], fitness: List[float],
                k: int, rng: random.Random) -> List[int]:
    """Draw k individuals at random; return a copy of the one with lowest cost."""
    candidates = rng.sample(range(len(population)), k)
    winner     = min(candidates, key=lambda i: fitness[i])
    return population[winner][:]

In [ ]:
# ── Canonical GA ───────────────────────────────────────────────────────────────
def ga_tsp(cost: np.ndarray,
           pop_size: int = 80,
           generations: int = 200,
           pc: float = 0.90,
           pm: Optional[float] = None,
           k_tournament: int = 3,
           elites: int = 1,
           mixed_init_frac: float = 0.15,
           crossover: str = "ox",
           seed: Optional[int] = None,
           record_history: bool = True):
    """
    Canonical Genetic Algorithm for TSP.

    Parameters
    ----------
    cost             : n×n distance matrix
    pop_size         : population size N
    generations      : number of generations G
    pc               : crossover probability (0.85–0.95 recommended)
    pm               : per-individual mutation probability; defaults to 1/n
    k_tournament     : tournament size k (3–5 typical)
    elites           : elite individuals copied unchanged each generation e
    mixed_init_frac  : fraction of population seeded with nearest-neighbour tours
    crossover        : "ox" (Order Crossover) or "pmx" (Partially Mapped)
    seed             : RNG seed
    record_history   : if True, record (generation, best_val) each generation

    Returns
    -------
    best_tour, best_val, history, elapsed_sec
    """
    rng = random.Random(seed)
    n   = cost.shape[0]
    if pm is None:
        pm = 1.0 / n           # slide recommendation: one mutation per tour

    cross_fn = crossover_ox if crossover == "ox" else crossover_pmx

    # ── Mixed initialisation (greedy NN + random) ────────────────────────────
    n_greedy = max(1, int(pop_size * mixed_init_frac))
    population: List[List[int]] = []
    for _ in range(n_greedy):
        start = rng.randint(0, n - 1)
        population.append(nearest_neighbor_tour(cost, start=start))
    for _ in range(pop_size - n_greedy):
        t = list(range(n))
        rng.shuffle(t)
        population.append(t)

    fitness   = [tour_length(cost, t) for t in population]
    best_idx  = min(range(pop_size), key=lambda i: fitness[i])
    best_tour = population[best_idx][:]
    best_val  = fitness[best_idx]
    history   = [(0, best_val)] if record_history else None
    t_start   = time.time()

    # ── Generational loop with elitism ───────────────────────────────────────
    for gen in range(1, generations + 1):
        # Identify elite individuals (sorted best-first)
        sorted_idx = sorted(range(pop_size), key=lambda i: fitness[i])
        new_pop    = [population[sorted_idx[i]][:] for i in range(elites)]

        # Fill remainder via selection + crossover + mutation
        while len(new_pop) < pop_size:
            p1 = _tournament(population, fitness, k_tournament, rng)
            p2 = _tournament(population, fitness, k_tournament, rng)

            if rng.random() < pc:          # crossover
                c1 = cross_fn(p1, p2, rng)
                c2 = cross_fn(p2, p1, rng)
            else:                          # clone (no crossover)
                c1, c2 = p1[:], p2[:]

            mutate_inversion(c1, pm, rng)  # inversion mutation
            mutate_inversion(c2, pm, rng)

            new_pop.append(c1)
            if len(new_pop) < pop_size:
                new_pop.append(c2)

        population = new_pop
        fitness    = [tour_length(cost, t) for t in population]

        gen_best = min(range(pop_size), key=lambda i: fitness[i])
        if fitness[gen_best] < best_val:
            best_val  = fitness[gen_best]
            best_tour = population[gen_best][:]

        if record_history:
            history.append((gen, best_val))

    return best_tour, best_val, history, time.time() - t_start

### GA convergence — same 40-city instance, fresh starting tour (seed 7)

We run all four methods on the same 40-city instance from the smoke test, but from a **fresh random starting tour** (seeded for reproducibility) that gives a clear separation between the two local-search pivots — about **8 %** gap between first- and best-improvement.  Each panel of the convergence plot uses its natural unit on the x-axis so that no curve is crushed by another's scale.

**What to look for:**

- **Panel 1 (LS):** best-pivot descends steadily to a noticeably lower cost in fewer passes than first-pivot. Because best-pivot always takes the single globally best 2-opt move available, each pass makes larger progress — at the cost of scanning the entire neighbourhood every time.
- **Panel 2 (SA):** starting from the same tour, SA's Metropolis acceptance lets it escape local optima that trap both LS variants, reaching a lower cost.
- **Panel 3 (GA):** the GA starts from an independently initialised population (not from `t0_ga`), so its x-axis counts generations, not evaluations. Rapid improvement in the first 50 generations reflects the spread of good building blocks; the plateau shows the population has converged. A pure GA without embedded local search typically lands *between* the two LS variants — crossover and population diversity help partially, but not as precisely as systematic neighbourhood search. This motivates the **Memetic GA** in Section 5.


In [ ]:
# ── Fresh starting tour for the GA convergence demo (seed 7) ─────────────────
# Using seed 7 gives an ~8 % gap between first- and best-improvement LS,
# making the pivot comparison clearly visible before introducing the GA.
set_seeds(7)
t0_ga = random_tour(len(C))   # reproducible starting tour (different from t0)

_, v1_ga, h1_ga, t1_ga = ls_first_improvement_2opt(C, t0_ga[:], record_history=True)
_, v2_ga, h2_ga, t2_ga = ls_best_improvement_2opt(C, t0_ga[:], record_history=True)
_, sa_val_ga, sa_hist_ga, sa_time_ga = sa_tsp(
    C, t0_ga[:], alpha=0.95, max_stages=300, seed=42)

ga_tour, ga_val, ga_hist, ga_time = ga_tsp(
    C, pop_size=80, generations=200, seed=42)

print("─" * 48)
print(f"{'Method':<26}  {'Cost':>8}  {'Time':>6}")
print("─" * 48)
print(f"{'First-impr. LS':<26}  {v1_ga:>8.2f}  {t1_ga:>5.3f}s")
print(f"{'Best-impr. LS':<26}  {v2_ga:>8.2f}  {t2_ga:>5.3f}s")
print(f"{'SA  (α=0.95, K=300)':<26}  {sa_val_ga:>8.2f}  {sa_time_ga:>5.3f}s")
print(f"{'GA  (N=80, G=200, OX)':<26}  {ga_val:>8.2f}  {ga_time:>5.3f}s")
print("─" * 48)
print(f"\nBest-pivot gain over first-pivot: {100*(v1_ga - v2_ga)/v1_ga:.1f} %")


In [ ]:
# Three-panel convergence plot — each method on its own natural x-axis scale.
# Putting LS (0–100 passes) and SA (0–120 000 evaluations) on the same axis
# compresses the LS curves to invisibility, so we give each panel its own scale.

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4))

# ── Panel 1: LS convergence (x = pass number) ───────────────────────────────
ax1.plot([h[0] for h in h1_ga], [h[1] for h in h1_ga],
         label="First-impr. LS", color="tab:blue", lw=1.8)
ax1.plot([h[0] for h in h2_ga], [h[1] for h in h2_ga],
         label="Best-impr. LS",  color="tab:green", lw=1.8)
ax1.axhline(v1_ga, ls="--", color="tab:blue",  lw=0.8, alpha=0.5)
ax1.axhline(v2_ga, ls="--", color="tab:green", lw=0.8, alpha=0.5)
ax1.set_xlabel("Pass (iteration)")
ax1.set_ylabel("Tour length")
ax1.set_title("LS convergence — first vs best pivot")
ax1.legend(fontsize=9)

# ── Panel 2: SA convergence (x = cumulative 2-opt evaluations) ──────────────
ax2.plot([h[0] for h in sa_hist_ga], [h[1] for h in sa_hist_ga],
         color="tab:orange", lw=1.5, label="SA (α=0.95)")
ax2.axhline(sa_val_ga, ls="--", color="tab:orange", lw=0.8, alpha=0.6)
# Reference lines for LS final values
ax2.axhline(v1_ga, ls=":", color="tab:blue",  lw=0.9, alpha=0.6, label=f"LS-first = {v1_ga:.0f}")
ax2.axhline(v2_ga, ls=":", color="tab:green", lw=0.9, alpha=0.6, label=f"LS-best  = {v2_ga:.0f}")
ax2.set_xlabel("Cumulative 2-opt evaluations  (stage × L)")
ax2.set_ylabel("Best-so-far tour length")
ax2.set_title("SA convergence  (K = 300 stages)")
ax2.legend(fontsize=8)

# ── Panel 3: GA convergence (x = generation) ────────────────────────────────
ax3.plot([h[0] for h in ga_hist], [h[1] for h in ga_hist],
         color="tab:red", lw=1.5, label="GA")
ax3.axhline(ga_val, ls="--", color="tab:red",   lw=0.8, alpha=0.6)
ax3.axhline(v1_ga,  ls=":", color="tab:blue",   lw=0.9, alpha=0.6, label=f"LS-first = {v1_ga:.0f}")
ax3.axhline(v2_ga,  ls=":", color="tab:green",  lw=0.9, alpha=0.6, label=f"LS-best  = {v2_ga:.0f}")
ax3.set_xlabel("Generation")
ax3.set_ylabel("Best-so-far tour length")
ax3.set_title("GA convergence  (N = 80, G = 200, OX)")
ax3.legend(fontsize=8)

plt.suptitle("40-city TSP — convergence comparison (seed-7 starting tour)", y=1.02)
plt.tight_layout()
plt.show()


### Exercises — Genetic Algorithm

**Exercise 3.1 — OX by hand.**
Given $P_1 = (A, B, C, D, E, F, G, H)$ and $P_2 = (C, F, A, H, B, D, G, E)$ with cut points $a = 3, b = 5$, perform OX by hand. Show the segment copied from P1, the scan order through P2, and the final child. Verify it is a valid permutation.

**Exercise 3.2 — PMX mapping chain.**
Using the same parents and cut points as Exercise 3.1, perform PMX. For each position outside $[3, 5]$, write out the complete mapping chain. Compare the resulting child to the OX child — which positions differ?

**Exercise 3.3 — Tournament size sweep.**
Run the GA on the 40-city instance with $k \in \{2, 3, 5, 7, 10\}$ (all other parameters fixed at defaults). Plot convergence curves. How does increasing $k$ affect (a) the speed of convergence and (b) the final solution quality? At what point does high selection pressure become harmful?

**Exercise 3.4 — Population size vs generations trade-off.**
With a fixed total budget of $N \times G = 12{,}000$ evaluations, try $(N, G) \in \{(30, 400), (60, 200), (120, 100), (240, 50)\}$. Which combination gives the best results? Why is there a sweet spot — what goes wrong with too few individuals or too few generations?

**Exercise 3.5 — OX vs PMX comparison.**
Run the GA 10 times with OX and 10 times with PMX (same instance, $n = 60$, same parameters otherwise). Compare the distributions of final tour lengths with a boxplot. Is there a statistically significant difference? (Hint: use a paired t-test or Wilcoxon signed-rank test.)

**Exercise 3.6 — Diversity tracking.**
Modify `ga_tsp` to record the *population diversity* at each generation, measured as the standard deviation of tour lengths in the population: $\sigma_g = \text{std}(\{f(\pi) : \pi \in P_g\})$. Plot diversity alongside the best fitness over generations. What happens to diversity as the GA converges? How does the mutation rate $p_m$ affect diversity loss?

**Exercise 3.7 — Crossover is key (ablation study).**
Run the GA with $p_c = 0$ (no crossover — offspring are just mutated clones) and compare to the default $p_c = 0.90$. How much worse is the GA without crossover? This demonstrates that recombination, not just selection + mutation, is the primary driver of GA performance.


### 5 · Memetic GA: GA + 2-opt local search

**The idea that changes everything.** A pure GA often produces offspring that are valid tours but contain obvious inefficiencies (crossing edges, detours). A simple 2-opt local search can remove these quickly. The **Memetic GA** (Moscato 1989; Mühlenbein 1992) applies local search to *every offspring* before inserting it into the population, so the GA operates entirely on locally optimal solutions.

**Why is this so effective?** Consider what crossover does: it combines subsequences from two good parents. The resulting child inherits good structure but may have "seam" defects where the spliced segments don't mesh well. A quick 2-opt pass smooths out these seams, producing a child that is often better than both parents.

The change to the canonical GA loop is minimal but transformative:

```
c1, c2 ← Crossover(p1, p2)
Mutate(c1, pm);  Mutate(c2, pm)
2-opt(c1)   ← NEW: polish offspring with local search
2-opt(c2)   ← NEW
P' ← P' ∪ {c1, c2}
```

**Performance in the literature:** Merz & Freisleben (1997) report that Memetic GA (OX + 2-opt) finds solutions within 0.3–2.8% of optimal on TSPLIB benchmarks, compared to 5–18% for pure GA. The local search overhead per offspring is modest (a few 2-opt passes), but the quality improvement is dramatic.

**Implementation note:** Instead of calling the full `ls_first_improvement_2opt` (which has overhead from history tracking and shuffling), we use a lean `_two_opt_pass` function that does a single deterministic first-improvement sweep through all pairs. The `ls_passes` parameter controls how many sweeps are applied — more passes mean solutions closer to local optimality but at higher computational cost.

**Trade-off: population size and generations.** Since each offspring evaluation is now much more expensive (due to local search), the Memetic GA typically uses a smaller population and fewer generations than the pure GA. This is compensated by the much higher quality of each individual.


In [ ]:
def _two_opt_pass(cost: np.ndarray, tour: List[int]) -> bool:
    """
    One first-improvement 2-opt pass (deterministic, no shuffling).
    Returns True if at least one improvement was made.
    Used inside the memetic GA inner loop for speed.
    """
    n = len(tour)
    for i in range(1, n - 1):
        for j in range(i + 1, n):
            d = two_opt_delta(cost, tour, i, j)
            if d < -1e-10:
                apply_two_opt(tour, i, j)
                return True          # first improvement → restart outer loop
    return False                     # local optimum


def memetic_ga_tsp(cost: np.ndarray,
                   pop_size: int = 60,
                   generations: int = 150,
                   pc: float = 0.90,
                   pm: Optional[float] = None,
                   k_tournament: int = 3,
                   elites: int = 1,
                   mixed_init_frac: float = 0.15,
                   ls_passes: int = 1,
                   seed: Optional[int] = None,
                   record_history: bool = True):
    """
    Memetic GA: canonical GA with 2-opt local search applied to every offspring.

    Parameters
    ----------
    ls_passes : number of first-improvement 2-opt passes applied to each
                offspring (1 = one sweep through all pairs; use ≥ 3 for
                near-local-optimality; full convergence if set very high)
    (all other parameters as in ga_tsp)
    """
    rng = random.Random(seed)
    n   = cost.shape[0]
    if pm is None:
        pm = 1.0 / n

    # ── Mixed initialisation + 2-opt each individual ─────────────────────────
    n_greedy = max(1, int(pop_size * mixed_init_frac))
    population: List[List[int]] = []
    for _ in range(n_greedy):
        t = nearest_neighbor_tour(cost, start=rng.randint(0, n - 1))
        for _ in range(ls_passes):
            if not _two_opt_pass(cost, t): break
        population.append(t)
    for _ in range(pop_size - n_greedy):
        t = list(range(n))
        rng.shuffle(t)
        for _ in range(ls_passes):
            if not _two_opt_pass(cost, t): break
        population.append(t)

    fitness   = [tour_length(cost, t) for t in population]
    best_idx  = min(range(pop_size), key=lambda i: fitness[i])
    best_tour = population[best_idx][:]
    best_val  = fitness[best_idx]
    history   = [(0, best_val)] if record_history else None
    t_start   = time.time()

    # ── Generational loop ────────────────────────────────────────────────────
    for gen in range(1, generations + 1):
        sorted_idx = sorted(range(pop_size), key=lambda i: fitness[i])
        new_pop    = [population[sorted_idx[i]][:] for i in range(elites)]

        while len(new_pop) < pop_size:
            p1 = _tournament(population, fitness, k_tournament, rng)
            p2 = _tournament(population, fitness, k_tournament, rng)

            if rng.random() < pc:
                c1 = crossover_ox(p1, p2, rng)
                c2 = crossover_ox(p2, p1, rng)
            else:
                c1, c2 = p1[:], p2[:]

            mutate_inversion(c1, pm, rng)
            mutate_inversion(c2, pm, rng)

            # ── Local search on offspring (the memetic step) ──────────────
            for _ in range(ls_passes):
                if not _two_opt_pass(cost, c1): break
            for _ in range(ls_passes):
                if not _two_opt_pass(cost, c2): break

            new_pop.append(c1)
            if len(new_pop) < pop_size:
                new_pop.append(c2)

        population = new_pop
        fitness    = [tour_length(cost, t) for t in population]

        gen_best = min(range(pop_size), key=lambda i: fitness[i])
        if fitness[gen_best] < best_val:
            best_val  = fitness[gen_best]
            best_tour = population[gen_best][:]

        if record_history:
            history.append((gen, best_val))

    return best_tour, best_val, history, time.time() - t_start


# ── Quick demo ────────────────────────────────────────────────────────────────
mga_tour, mga_val, mga_hist, mga_time = memetic_ga_tsp(
    C, pop_size=60, generations=150, ls_passes=3, seed=42)

print("─" * 52)
print(f"{'Method':<30}  {'Cost':>8}  {'Time':>6}")
print("─" * 52)
for label, val, t_el in [
    ("First-impr. LS",      v1,      t1),
    ("Best-impr. LS",       v2,      t2),
    ("SA  (α=0.95, K=300)", sa_val,  sa_time),
    ("GA  (N=80, G=200)",   ga_val,  ga_time),
    ("Memetic GA (N=60, G=150, LS=3)", mga_val, mga_time),
]:
    print(f"{label:<30}  {val:>8.2f}  {t_el:>5.3f}s")
print("─" * 52)

# Convergence plot: GA vs Memetic GA (both by generation)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot([h[0] for h in ga_hist],  [h[1] for h in ga_hist],  label="Canonical GA  (N=80)",   color="tab:red")
ax.plot([h[0] for h in mga_hist], [h[1] for h in mga_hist], label="Memetic GA   (N=60, LS=3)", color="tab:purple", lw=1.5)
ax.set_xlabel("Generation")
ax.set_ylabel("Best-so-far tour length")
ax.set_title("Canonical GA vs Memetic GA (40 cities)")
ax.legend()
plt.tight_layout()
plt.show()


### Full comparison: LS vs SA vs GA vs Memetic GA

This is the culminating experiment: we pit all six methods against each other on the same random instances, with 5 independent trials per problem size ($n \in \{40, 60\}$). This gives us a statistical picture of each method's performance — not just a single lucky or unlucky run.

**Methods compared:**
1. **LS-first** — single-start first-improvement 2-opt
2. **LS-best** — single-start best-improvement 2-opt
3. **SA ($\times$1)** — single-start simulated annealing ($\alpha = 0.95$, $K = 300$)
4. **SA-ms ($\times$3)** — multi-start SA with 3 independent runs
5. **GA** — canonical genetic algorithm ($N = 60$, $G = 150$, OX)
6. **Memetic GA** — GA + 2-opt local search ($N = 50$, $G = 100$, 3 LS passes per offspring)

**What to expect:** The Memetic GA should consistently produce the best (lowest) tour lengths, followed by SA-ms. Pure GA and single SA should be competitive but with higher variance. The two LS methods form the baseline — fast but limited by their inability to escape local optima.

The summary table shows mean, best, and worst tour length across the 5 trials. The boxplot provides a visual comparison of the distributions.


In [ ]:
def run_full_comparison(ns=(40, 60), trials=5, seed=123) -> pd.DataFrame:
    """
    Compare six methods across problem sizes and trials.
    All methods receive the same random instance per (n, trial) pair.
    """
    set_seeds(seed)
    rows = []
    for n in ns:
        for tr in range(trials):
            pts_e = generate_euclidean_points(n, seed=seed + tr * 11)
            Ce    = euclidean_dist_matrix(pts_e)
            init  = random_tour(n)        # shared starting tour for LS

            # ── Local Search ──────────────────────────────────────────────
            _, val, _, t = ls_first_improvement_2opt(Ce, init[:], record_history=False)
            rows.append(dict(n=n, trial=tr, method="LS-first",  cost=val, time=t))

            _, val, _, t = ls_best_improvement_2opt(Ce, init[:], record_history=False)
            rows.append(dict(n=n, trial=tr, method="LS-best",   cost=val, time=t))

            # ── SA (single start) ─────────────────────────────────────────
            _, val, _, t = sa_tsp(Ce, init[:], alpha=0.95,
                                  max_stages=300, seed=seed + tr)
            rows.append(dict(n=n, trial=tr, method="SA (×1)",   cost=val, time=t))

            # ── SA multi-start (3 runs) ───────────────────────────────────
            _, val, t = multistart_sa(Ce, runs=3, alpha=0.95,
                                      max_stages=200, seed=seed + tr)
            rows.append(dict(n=n, trial=tr, method="SA-ms (×3)", cost=val, time=t))

            # ── Canonical GA ──────────────────────────────────────────────
            _, val, _, t = ga_tsp(Ce, pop_size=60, generations=150,
                                  seed=seed + tr, record_history=False)
            rows.append(dict(n=n, trial=tr, method="GA",         cost=val, time=t))

            # ── Memetic GA ────────────────────────────────────────────────
            _, val, _, t = memetic_ga_tsp(Ce, pop_size=50, generations=100,
                                          ls_passes=3, seed=seed + tr,
                                          record_history=False)
            rows.append(dict(n=n, trial=tr, method="Memetic GA", cost=val, time=t))

    return pd.DataFrame(rows)


df_all = run_full_comparison(ns=(40, 60), trials=5)

summary = (df_all.groupby(["n", "method"])["cost"]
                 .agg(mean="mean", best="min", worst="max")
                 .round(1))
print(summary.to_string())

In [ ]:

# Box-plot: solution quality by method and problem size
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=False)
method_order = ["LS-first", "LS-best", "SA (×1)", "SA-ms (×3)", "GA", "Memetic GA"]
colors_box = ["tab:blue", "tab:green", "tab:orange", "tab:red", "tab:purple", "tab:brown"]
for ax, n_val in zip(axes, [40, 60]):
    sub  = df_all[df_all["n"] == n_val]
    data = [sub[sub["method"] == m]["cost"].values for m in method_order]
    bp   = ax.boxplot(data, tick_labels=method_order, patch_artist=True, sym="+")
    for patch, col in zip(bp["boxes"], colors_box):
        patch.set_facecolor(col); patch.set_alpha(0.6)
    ax.set_title(f"n = {n_val} cities  (5 trials)")
    ax.set_ylabel("Tour length" if n_val == 40 else "")
    plt.setp(ax.get_xticklabels(), rotation=25, ha="right", fontsize=7)
plt.suptitle("LS vs SA vs GA vs Memetic GA  (lower is better)", y=1.01)
plt.tight_layout(); plt.show()


### Exercises — Memetic GA and Final Comparison

**Exercise 4.1 — LS passes per offspring.**
Run the Memetic GA on the 40-city instance with `ls_passes` $\in \{1, 3, 5, 10, 50\}$. Plot (a) the final best tour length and (b) the total wall-clock time vs number of passes. Is there a diminishing-returns point? What does `ls_passes=50` effectively do to each offspring?

**Exercise 4.2 — Memetic GA vs multi-start LS: who wins?**
The Memetic GA applies local search to each offspring, which is similar to multi-start LS. But the GA also uses crossover to create the starting points for LS, rather than random tours. Design an experiment that isolates this effect: compare (a) Memetic GA ($N = 50$, $G = 100$) to (b) multi-start LS with $50 \times 100 = 5{,}000$ independent random restarts. Both use roughly the same total 2-opt budget. Which one wins, and by how much?

**Exercise 4.3 — Scaling study.**
Extend `run_full_comparison` to test $n \in \{40, 60, 80, 100\}$. Plot the mean tour length vs $n$ for all six methods. How does the gap between methods change as $n$ grows? Which method scales best?

**Exercise 4.4 — Time-limited comparison.**
In practice, we often have a fixed wall-clock budget rather than a fixed iteration count. Modify the experiment to give each method exactly 10 seconds per instance (for $n = 80$). Within that budget, each method can use its time however it wants (more restarts, more generations, etc.). Which method makes the best use of a fixed time budget?

**Exercise 4.5 — Hybrid SA+GA (advanced).**
Design a hybrid that uses SA instead of 2-opt for the local search step in the Memetic GA. Each offspring is improved by a short SA run ($K = 50$ stages) rather than deterministic 2-opt. Compare this SA-Memetic GA to the standard 2-opt-Memetic GA. Does the stochastic LS help the GA escape to better regions?

**Exercise 4.6 — TSPLIB benchmarks (advanced).**
Download a small TSPLIB instance (e.g., `eil51` with 51 cities, optimal tour length = 426). Parse the instance file, run all six methods, and report the percentage gap from the known optimum: $\text{gap} = 100 \cdot (f_{\text{found}} - f^*) / f^*$. How close does the Memetic GA get?


---
## Summary: The Metaheuristic Hierarchy

| Method | Mechanism for escaping local optima | Key strength | Key weakness |
|--------|-------------------------------------|-------------|-------------|
| **Local Search (2-opt)** | None — stops at the first local optimum found | Fast, simple, deterministic | Trapped in local optima |
| **Multi-start LS** | Multiple independent starts | Simple parallelism | No information sharing between runs |
| **Simulated Annealing** | Accepts uphill moves with decreasing probability | Theoretically converges to global optimum (with infinite time) | Single trajectory; sensitive to cooling schedule |
| **Genetic Algorithm** | Population diversity + crossover recombination | Parallel exploration; builds on good sub-structures | Solutions may contain obvious local defects |
| **Memetic GA** | GA's population search + LS polish on every offspring | Best of both worlds; state-of-the-art for many TSP sizes | Most expensive per evaluation; parameter tuning |

**The key takeaway:** each method builds on the limitations of the previous one. Local search is the foundation; SA adds escape mechanisms; GA adds population-level exploration; and the Memetic GA combines evolutionary search with local optimisation for consistently superior results.
